# OpenSky Live Ingestion

Fetches global state vectors from the OpenSky REST API for the **last 30 complete minutes**
(one API call per minute), and saves the results to the local PostgreSQL `oc-database`.

## Window Definition
- `now` is floored to the nearest UTC minute, then shifted back one minute to avoid the
  in-progress current minute (which may have incomplete data on OpenSky's side).
- The 30-minute window is `[now_floor - 30min, now_floor - 1min]` inclusive — always 30
  complete, settled minutes.

## Rate Limiting
A random 10–15 ms courtesy delay is applied **after** each API call (skipped after the last)
to avoid hammering the OpenSky API.

## Requirements
- A `credentials.json` file with `clientId` and `clientSecret` from your OpenSky API client.
- Authentication uses OAuth2 client credentials (Bearer token), which unlocks historical `?time=` queries.

## Schema Compatibility Note
The `opensky_requests` and `opensky_states` tables are designed to be forward-compatible
with Trino-based historical ingestion. When using Trino for a full hour, `minute` in
`opensky_requests` would be set to `0` (top of the hour), and `time_ts` in
`opensky_states` would store the Unix timestamp of each state vector.

## Configuration

In [ ]:
# --- User Configuration ---
CREDENTIALS_PATH = "../local_only/credentials.json"  # Path to your OpenSky credentials.json

## Imports

In [ ]:
import json
import random
import time
from datetime import datetime, timezone, timedelta
from pathlib import Path

import pandas as pd
import psycopg2
import psycopg2.extras
import requests
from tqdm.notebook import tqdm

## Database Connection

In [ ]:
DB_PARAMS = {
    "host": "localhost",
    "user": "leon",
    "password": "leon",
    "database": "oc-database",
    "port": 5432,
}

## Step 1 — Load Credentials

In [ ]:
TOKEN_URL = "https://auth.opensky-network.org/auth/realms/opensky-network/protocol/openid-connect/token"

creds_path = Path(CREDENTIALS_PATH)
assert creds_path.exists(), f"Credentials file not found: {creds_path.resolve()}"

with open(creds_path) as f:
    creds = json.load(f)

assert "clientId" in creds and "clientSecret" in creds, \
    "credentials.json must contain 'clientId' and 'clientSecret'"


class TokenManager:
    """Fetches and auto-refreshes an OAuth2 Bearer token using client credentials."""

    def __init__(self, client_id: str, client_secret: str):
        self._client_id = client_id
        self._client_secret = client_secret
        self._token: str | None = None
        self._expires_at: float = 0.0

    def _refresh(self) -> None:
        r = requests.post(TOKEN_URL, data={
            "grant_type": "client_credentials",
            "client_id": self._client_id,
            "client_secret": self._client_secret,
        })
        r.raise_for_status()
        data = r.json()
        self._token = data["access_token"]
        self._expires_at = time.time() + data.get("expires_in", 1800) - 30
        print(f"Bearer token acquired (expires in ~{data.get('expires_in', 1800)}s)")

    @property
    def headers(self) -> dict:
        if not self._token or time.time() >= self._expires_at:
            self._refresh()
        return {"Authorization": f"Bearer {self._token}"}


token_manager = TokenManager(creds["clientId"], creds["clientSecret"])
print(f"Loaded credentials for clientId: {creds['clientId']}")

## Step 2 — Compute 30-Minute Window

In [ ]:
MINUTES = 30
REST_URL = "https://opensky-network.org/api/states/all"

# Floor to the nearest UTC minute, then back one minute to avoid the in-progress minute
now = datetime.now(timezone.utc)
window_end = now.replace(second=0, microsecond=0) - timedelta(minutes=1)
window_start = window_end - timedelta(minutes=MINUTES - 1)

# Derive request metadata from window start
DATE   = window_start.strftime("%Y-%m-%d")
HOUR   = window_start.hour
MINUTE = window_start.minute  # start minute of the window (0-59)
WINDOW_START_TS = int(window_start.timestamp())

# Build the list of Unix timestamps to fetch (one per minute, inclusive)
timestamps = [
    int((window_start + timedelta(minutes=i)).timestamp())
    for i in range(MINUTES)
]

print(f"Window : {window_start.isoformat()} -> {window_end.isoformat()} UTC")
print(f"Request: date={DATE}, hour={HOUR}, minute={MINUTE}")
print(f"Timestamps: {timestamps[0]} ... {timestamps[-1]} ({len(timestamps)} total)")

## Step 3 — Fetch State Vectors from REST API

> One request per minute. Fails fast on any HTTP error.
> A random 10–15 ms delay is applied after each call (skipped after the last).

In [ ]:
COLUMNS = [
    "icao24", "callsign", "origin_country", "time_position", "last_contact",
    "lon", "lat", "baro_altitude", "on_ground", "velocity",
    "true_track", "vertical_rate", "sensors", "geo_altitude",
    "squawk", "spi", "position_source",
]

all_rows = []

for i, ts in enumerate(tqdm(timestamps, desc="Fetching minutes", unit="min")):
    resp = requests.get(
        REST_URL,
        params={"time": ts},
        headers=token_manager.headers,
        timeout=30,
    )
    resp.raise_for_status()

    states = resp.json().get("states") or []
    for state in states:
        row = dict(zip(COLUMNS, state))
        row["time_ts"] = ts
        all_rows.append(row)

    # Random 10-15 ms courtesy delay — skipped after the last call
    if i < len(timestamps) - 1:
        time.sleep(random.uniform(0.010, 0.015))

df = pd.DataFrame(all_rows)

if df.empty:
    raise ValueError("No data returned from REST API. Check credentials and window range.")

print(f"Fetched {len(df):,} state vectors across {df['time_ts'].nunique()} timestamps.")
df.head()

## Step 4 — Deduplicate to Latest State per Aircraft per Timestamp

In [ ]:
df = (
    df.sort_values("time_ts")
    .groupby(["time_ts", "icao24"], as_index=False)
    .last()
)

print(f"{len(df):,} state vectors across {df['time_ts'].nunique()} timestamps after deduplication.")
print(f"Timestamp range: {df['time_ts'].min()} - {df['time_ts'].max()}")

## Step 5 — Set Up PostgreSQL Tables

In [ ]:
pg_conn = psycopg2.connect(**DB_PARAMS)
pg_conn.autocommit = False

with pg_conn.cursor() as cur:
    cur.execute("CREATE EXTENSION IF NOT EXISTS pgcrypto;")
    cur.execute("CREATE EXTENSION IF NOT EXISTS postgis;")

    # Registry table: one row per ingestion run.
    # UNIQUE(date, hour, minute) identifies the start of each window.
    # Trino compatibility note: for full-hour historical runs, minute = 0.
    cur.execute("""
        CREATE TABLE IF NOT EXISTS opensky_requests (
            id               UUID PRIMARY KEY DEFAULT gen_random_uuid(),
            date             DATE NOT NULL,
            hour             INT NOT NULL,
            minute           INT NOT NULL,
            window_start_ts  BIGINT NOT NULL,
            created_at       TIMESTAMP DEFAULT NOW(),
            UNIQUE (date, hour, minute)
        );
    """)

    # State vectors table.
    # time_ts: Unix timestamp of the API snapshot this state belongs to.
    # geom: PostGIS point geometry (WGS84) — lon/lat stored as GEOMETRY(Point, 4326).
    cur.execute("""
        CREATE TABLE IF NOT EXISTS opensky_states (
            id            UUID PRIMARY KEY DEFAULT gen_random_uuid(),
            request_id    UUID REFERENCES opensky_requests(id) ON DELETE CASCADE,
            time_ts       BIGINT NOT NULL,
            icao24        TEXT,
            callsign      TEXT,
            geom          GEOMETRY(Point, 4326),
            velocity      DOUBLE PRECISION,
            heading       DOUBLE PRECISION,
            vertrate      DOUBLE PRECISION,
            baro_altitude DOUBLE PRECISION,
            geo_altitude  DOUBLE PRECISION,
            on_ground     BOOLEAN,
            squawk        TEXT,
            UNIQUE (request_id, time_ts, icao24)
        );
    """)

    cur.execute("CREATE INDEX IF NOT EXISTS idx_opensky_states_request_id ON opensky_states (request_id);")
    cur.execute("CREATE INDEX IF NOT EXISTS idx_opensky_states_time_ts ON opensky_states (time_ts);")
    cur.execute("CREATE INDEX IF NOT EXISTS idx_opensky_states_geom ON opensky_states USING GIST (geom);")

pg_conn.commit()
print("Tables ready.")

## Step 6 — Register the Request

> Raises a `psycopg2.errors.UniqueViolation` if a run for this window already exists.
> Delete the existing row from `opensky_requests` to re-ingest.

In [ ]:
with pg_conn.cursor() as cur:
    cur.execute("""
        INSERT INTO opensky_requests (date, hour, minute, window_start_ts)
        VALUES (%s, %s, %s, %s)
        RETURNING id;
    """, (DATE, HOUR, MINUTE, WINDOW_START_TS))
    REQUEST_ID = cur.fetchone()[0]

pg_conn.commit()
print(f"Registered new request: {REQUEST_ID}")
print(f"  date={DATE}, hour={HOUR}, minute={MINUTE}, window_start_ts={WINDOW_START_TS}")

## Step 7 — Insert State Vectors

In [ ]:
ts_values = sorted(df["time_ts"].unique())

with pg_conn.cursor() as cur:
    for ts in tqdm(ts_values, desc="Inserting timestamps", unit="ts"):
        ts_df = df[df["time_ts"] == ts]

        records = []
        for _, row in ts_df.iterrows():
            lat = float(row["lat"]) if pd.notna(row.get("lat")) else None
            lon = float(row["lon"]) if pd.notna(row.get("lon")) else None
            geom = f"SRID=4326;POINT({lon} {lat})" if lat is not None and lon is not None else None
            records.append((
                str(REQUEST_ID),
                int(ts),
                row.get("icao24") or None,
                row["callsign"].strip() if pd.notna(row.get("callsign")) else None,
                geom,
                float(row["velocity"])      if pd.notna(row.get("velocity"))      else None,
                float(row["true_track"])    if pd.notna(row.get("true_track"))    else None,
                float(row["vertical_rate"]) if pd.notna(row.get("vertical_rate")) else None,
                float(row["baro_altitude"]) if pd.notna(row.get("baro_altitude")) else None,
                float(row["geo_altitude"])  if pd.notna(row.get("geo_altitude"))  else None,
                bool(row["on_ground"])      if pd.notna(row.get("on_ground"))     else None,
                row.get("squawk") or None,
            ))

        psycopg2.extras.execute_values(
            cur,
            """
            INSERT INTO opensky_states
                (request_id, time_ts, icao24, callsign, geom,
                 velocity, heading, vertrate, baro_altitude, geo_altitude,
                 on_ground, squawk)
            VALUES %s
            ON CONFLICT (request_id, time_ts, icao24) DO NOTHING;
            """,
            records,
        )

    pg_conn.commit()

print(f"Done. {len(df):,} state vectors inserted across {len(ts_values)} timestamps.")

## Step 8 — Verify

In [ ]:
with pg_conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor) as cur:
    cur.execute("""
        SELECT r.id, r.date, r.hour, r.minute, r.window_start_ts, r.created_at,
               COUNT(s.id)               AS state_count,
               COUNT(DISTINCT s.time_ts) AS timestamps_covered,
               COUNT(DISTINCT s.icao24)  AS unique_aircraft
        FROM opensky_requests r
        LEFT JOIN opensky_states s ON s.request_id = r.id
        WHERE r.id = %s
        GROUP BY r.id, r.date, r.hour, r.minute, r.window_start_ts, r.created_at;
    """, (str(REQUEST_ID),))
    summary = cur.fetchone()

print("Ingestion summary:")
for k, v in summary.items():
    print(f"  {k}: {v}")

pg_conn.close()